# Inference

## Imports and Setup

In [90]:
import numpy as np
import pandas as pd
import mlflow
import dagshub

dagshub.init(repo_owner="NikaMikeltadze", repo_name="assignment_1_nmike23", mlflow=True)
mlflow.set_experiment('house-prices-contest')

Initialized MLflow to track repo "NikaMikeltadze/assignment_1_nmike23"

Repository NikaMikeltadze/assignment_1_nmike23 initialized!

<Experiment: artifact_location='mlflow-artifacts:/0113ddf519354c19897c70c3f344207f', creation_time=1775509743209, experiment_id='0', last_update_time=1775509743209, lifecycle_stage='active', name='house-prices-contest', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

## Load Data and Proccess as in model training

In [91]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

# Drop extreme outliers
if 'SalePrice' in train_data.columns:
    train_data = train_data.drop(train_data[(train_data['GrLivArea'] > 4000) & (train_data['SalePrice'] < 300000)].index).reset_index(drop=True)

# Drop columns id and high NA
train_data = train_data.drop(columns=['Id'])
na_stat = train_data.isna().mean().sort_values(ascending=False).head(10)
na_rm_cols = na_stat[na_stat > 0.4].index
train_data = train_data.drop(columns=na_rm_cols, errors='ignore')

# Fill NA values
not_present_cols = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                    'FireplaceQu', 'GarageType', 'GarageFinish',
                    'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']

for col in not_present_cols:
    if col in train_data.columns:
        train_data[col] = train_data[col].fillna('NP')
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna('NP')

quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NP': 0}
quality_numeric_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]
for col in quality_numeric_cols:
    if col in train_data.columns:
        train_data[col] = train_data[col].fillna('NP').map(quality_map)
        test_data[col] = test_data[col].fillna('NP').map(quality_map)

type_numeric_cols = train_data.select_dtypes(include=[np.number]).columns
for col in type_numeric_cols:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(train_data[col].median())

type_category_cols = train_data.select_dtypes(include=['object']).columns
for col in type_category_cols:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(train_data[col].mode()[0])

# Feature Engineering
train_data['TotalSF'] = train_data['TotalBsmtSF'] + train_data['1stFlrSF'] + train_data['2ndFlrSF']
test_data['TotalSF'] = test_data['TotalBsmtSF'] + test_data['1stFlrSF'] + test_data['2ndFlrSF']

train_data['TotalLivingSF'] = train_data['GrLivArea'] + train_data['TotalBsmtSF']
test_data['TotalLivingSF'] = test_data['GrLivArea'] + test_data['TotalBsmtSF']

train_data['TotalBathrooms'] = (train_data['FullBath'] + (0.5 * train_data['HalfBath']) + train_data['BsmtFullBath'] + (
            0.5 * train_data['BsmtHalfBath']))
test_data['TotalBathrooms'] = (test_data['FullBath'] + (0.5 * test_data['HalfBath']) + test_data['BsmtFullBath'] + (
            0.5 * test_data['BsmtHalfBath']))

train_data['BedroomRatio'] = train_data['BedroomAbvGr'] / (train_data['TotRmsAbvGrd'])
train_data['KitchenRatio'] = train_data['KitchenAbvGr'] / (train_data['TotRmsAbvGrd'])
train_data['RoomRatio'] = train_data['TotRmsAbvGrd'] / (train_data['TotalSF'])

test_data['BedroomRatio'] = test_data['BedroomAbvGr'] / (test_data['TotRmsAbvGrd'])
test_data['KitchenRatio'] = test_data['KitchenAbvGr'] / (test_data['TotRmsAbvGrd'])
test_data['RoomRatio'] = test_data['TotRmsAbvGrd'] / (test_data['TotalSF'])

train_data['HouseAge'] = train_data['YrSold'] - train_data['YearBuilt']
train_data['RemodAge'] = train_data['YrSold'] - train_data['YearRemodAdd']
train_data['WasRemodeled'] = (train_data['YearRemodAdd'] != train_data['YearBuilt']).astype(int)

test_data['HouseAge'] = test_data['YrSold'] - test_data['YearBuilt']
test_data['RemodAge'] = test_data['YrSold'] - test_data['YearRemodAdd']
test_data['WasRemodeled'] = (test_data['YearRemodAdd'] != test_data['YearBuilt']).astype(int)

train_data['HasPool'] = (train_data['PoolArea'] > 0).astype(int)
train_data['HasGarage'] = (train_data['GarageArea'] > 0).astype(int)
train_data['HasFireplace'] = (train_data['Fireplaces'] > 0).astype(int)
train_data['HasBasement'] = (train_data['TotalBsmtSF'] > 0).astype(int)

test_data['HasPool'] = (test_data['PoolArea'] > 0).astype(int)
test_data['HasGarage'] = (test_data['GarageArea'] > 0).astype(int)
test_data['HasFireplace'] = (test_data['Fireplaces'] > 0).astype(int)
test_data['HasBasement'] = (test_data['TotalBsmtSF'] > 0).astype(int)

train_data['QualityScore'] = train_data['OverallQual'] * train_data['OverallCond'] * train_data['TotalSF']
test_data['QualityScore'] = test_data['OverallQual'] * test_data['OverallCond'] * test_data['TotalSF']

train_data['GarageQualityScore'] = train_data['GarageQual'] * train_data['GarageCond'] * train_data['GarageArea']
test_data['GarageQualityScore'] = test_data['GarageQual'] * test_data['GarageCond'] * test_data['GarageArea']

X_train = train_data.drop(columns=['SalePrice'])
Y_train = np.log1p(train_data['SalePrice'])

# Apply log transformation to highly skewed continuous features
skewed_cols = ['LotArea', '1stFlrSF', 'GrLivArea', 'TotalBsmtSF', 'TotalSF', 'TotalLivingSF']
for col in skewed_cols:
    if col in X_train.columns:
        X_train[col] = np.log1p(X_train[col])
    if col in test_data.columns:
        test_data[col] = np.log1p(test_data[col])

categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=[np.number]).columns

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
test_ids = test_data['Id']
X_test = pd.get_dummies(test_data.drop(columns=['Id']), columns=categorical_cols, drop_first=True)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

X_train_temp = X_train.copy()
X_train_temp['SalePrice'] = Y_train
target_corr = X_train_temp.corr()['SalePrice'].abs()

low_corr_features = target_corr[target_corr < 0.05].index.tolist()
low_corr_features = [f for f in low_corr_features if f != 'SalePrice' and f in X_train.columns]

X_train = X_train.drop(columns=low_corr_features)
X_test = X_test.drop(columns=low_corr_features)

X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(X_train.median())

print(f"X_test shape: {X_test.shape}")

X_test shape: (1459, 134)


## Load Best Model from MLflow Registry

In [92]:
# Pick Best Performing Model
model_name = "HousePrices_Ridge_Optimized"
model_version = "5"
logged_model = f'models:/{model_name}/{model_version}'

best_model = mlflow.pyfunc.load_model(logged_model)

## Make Predictions and Generate Submission

In [93]:
predictions_log = best_model.predict(X_test)
predictions_orig = np.expm1(predictions_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions_orig
})

submission.to_csv('submission.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,118308.107956
1,1462,166002.860925
2,1463,182598.204876
3,1464,198832.622426
4,1465,207864.969267


In [94]:
!kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission.csv -m "Ridge_Optimized v5 model from MLflow"

400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission



  0%|          | 0.00/35.1k [00:00<?, ?B/s]
 46%|████▌     | 16.0k/35.1k [00:00<00:00, 29.3kB/s]
100%|██████████| 35.1k/35.1k [00:01<00:00, 31.5kB/s]


In [95]:
!kaggle competitions submissions -c house-prices-advanced-regression-techniques

fileName        date                        description                                  status                     publicScore  privateScore  
--------------  --------------------------  -------------------------------------------  -------------------------  -----------  ------------  
submission.csv  2026-04-08 13:10:40.623000  RandomForest_Optimized v5 model from MLflow  SubmissionStatus.COMPLETE  0.14026                    
submission.csv  2026-04-08 13:09:39.347000  DecisionTree_Optimized v5 model from MLflow  SubmissionStatus.COMPLETE  0.17486                    
submission.csv  2026-04-08 13:08:37.930000  Ridge_Optimized v5 model from MLflow         SubmissionStatus.COMPLETE  0.13837                    
submission.csv  2026-04-08 12:47:34.700000  RandomForestRegressor v6 model from MLflow   SubmissionStatus.COMPLETE  0.15190                    
submission.csv  2026-04-08 12:40:10.933000  RandomForestRegressor model from MLflow      SubmissionStatus.COMPLETE  0.15190             